In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df=spark.read.format("parquet")\
    .load("abfss://bronze@databricksetestorage.dfs.core.windows.net/orders")

In [0]:
display(df)

In [0]:
df.printSchema()

### **Data Enrichment**

In [0]:
df=df.withColumnRenamed("_rescued_data","rescued_data")

In [0]:
df=df.drop("rescued_data")
df.display()

In [0]:
df=df.withColumn("order_date",to_timestamp(col("order_date")))
df.display()

In [0]:
df=df.withColumn("year",year(col("order_date")))
df.display()

In [0]:
df1=df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df1.display()

In [0]:
df1=df1.withColumn("rank_flag",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df1.display()

In [0]:
df1=df1.withColumn("row_flag",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df1.display()

### **Class - OOP**

In [0]:
class Windows:
    def __init__(self,df):
        self.df=df
    def dense_rank(self):
        df_dense_rank = self.df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank    

In [0]:
df_new=df
df_new.display()

In [0]:
obj=Windows(df_new)

In [0]:
obj.dense_rank()

In [0]:
df_res=obj.dense_rank()
df_res.display()

### **Data Writing**

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@databricksetestorage.dfs.core.windows.net/orders")

In [0]:
%sql
create table if not exists databricks_cat.silver.orders_silver
using delta
location "abfss://silver@databricksetestorage.dfs.core.windows.net/orders"